SETUP

In [26]:
import os
import time
import uuid
import json
from dotenv import load_dotenv

In [27]:
from portkey_ai import Portkey, createHeaders, PORTKEY_GATEWAY_URL

In [28]:
load_dotenv(dotenv_path="../.env")
PORTKEY_API_KEY=os.getenv("PORTKEY_API_KEY","oTp81rbDw30yq5fK8FEDAu6LoQLM")
portkey= Portkey(api_key=PORTKEY_API_KEY)

SETUP VIRTUAL KEYS AND SLUGS

In [29]:
GROQ_SLUG="gkey"
GROQ_MODEL=f"@{GROQ_SLUG}/openai/gpt-oss-120b"

GROQ_SLUG_2="gkey"
GROQ_MODEL_2=f"@{GROQ_SLUG_2}/llama-3.1-8b-instant"

GROQ_API_KEY=os.getenv("GROQ_API_KEY")

In [30]:
print("Setup complete!")
print(f"  Portkey API Key : {'OK' if PORTKEY_API_KEY else 'MISSING'}")
print(f"  Groq slug       : {GROQ_SLUG}")
print(f"  Groq model ref  : {GROQ_MODEL}")
print(f"  Groq slug 2     : {GROQ_SLUG_2}")
print(f"  Small model ref : {GROQ_MODEL_2}")
print(f"\nPortkey Gateway : {PORTKEY_GATEWAY_URL}")

Setup complete!
  Portkey API Key : OK
  Groq slug       : gkey
  Groq model ref  : @gkey/openai/gpt-oss-120b
  Groq slug 2     : gkey
  Small model ref : @gkey/llama-3.1-8b-instant

Portkey Gateway : https://api.portkey.ai/v1


HELPER FUNCTION

In [7]:
## To print the title
def section(title):
    print(f"\n{"="*65}")
    print(f"{title}")
    print(f"{"="*65}")

section("Abhay")



Abhay


RESPONSE

In [8]:
def show(q, answer, ms, label=""):
    bar = chr(9472) * 62
    print(f"\n{bar}")
    print(f"Q: {q}")
    print(f"A: {answer[:260]}{'...' if len(answer) > 260 else ''}")
    note = f" | {label}" if label else ""
    print(f"⏱  {ms:.0f}ms{note}")
    print(bar)

In [9]:
## it is just to print god nothing much
show("why is the sky blue ?","nature",5,"god")


──────────────────────────────────────────────────────────────
Q: why is the sky blue ?
A: nature
⏱  5ms | god
──────────────────────────────────────────────────────────────


---
### BASELINE — Direct LLM Call (No Gateway)

**Goal:** See what a raw LLM call looks like — no routing, no logging, no resilience.

In [10]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

In [11]:
raw_groq = ChatGroq(api_key=GROQ_API_KEY, model="openai/gpt-oss-120b", temperature=0)

section("BASELINE — Direct Groq Call")

questions = [
    "What is Kubernetes in one sentence?",
    "What is Intel SRIOV?",
]

for q in questions:
    t0 = time.time()
    r = raw_groq.invoke([HumanMessage(content=q)])
    show(q, r.content, (time.time()-t0)*1000, label="direct Groq — no gateway")



BASELINE — Direct Groq Call

──────────────────────────────────────────────────────────────
Q: What is Kubernetes in one sentence?
A: Kubernetes is an open‑source platform that automates the deployment, scaling, and management of containerized applications across clusters of machines.
⏱  1166ms | direct Groq — no gateway
──────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────
Q: What is Intel SRIOV?
A: **Intel SR‑IOV (Single‑Root I/O Virtualization)** is a hardware‑level technology that lets a single physical network (or other PCIe) device appear as multiple independent virtual devices. Each virtual device—called a **Virtual Function (VF)**—can be assigned d...
⏱  5080ms | direct Groq — no gateway
──────────────────────────────────────────────────────────────


---
# EXPERIMENT 1 — Route Through the Gateway

**Goal:** Same call, same answer — but now every request is logged in your Portkey dashboard.

**New concepts:**
- `Portkey(api_key=...)` — the gateway client
- `model="@slug/model-name"` — tells Portkey which provider to use
- `response.choices[0].message.content` — standard OpenAI response format

In [12]:
section("EXP 1 — Basic Gateway Call")

questions = [
    "What is AI and gen ai ?",
    "What is coffee and black coffee?",
]


for q in questions:
    t0 = time.time()
    r = portkey.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role":"user","content":q}]
    )
    show(q, r.choices[0].message.content, (time.time()-t0)*1000,
         label="routed via Portkey gateway")
    
print("\n✅ Check portkey.ai → Logs to see both requests fully logged!")
print("   Token count, cost, latency — all tracked. Zero extra code.")


EXP 1 — Basic Gateway Call

──────────────────────────────────────────────────────────────
Q: What is AI and gen ai ?
A: **Artificial Intelligence (AI)**  
Artificial Intelligence is a branch of computer science that aims to create machines (software, hardware, or both) that can perform tasks that normally require human intelligence. Those tasks include things like:

| Human‑lik...
⏱  4860ms | routed via Portkey gateway
──────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────
Q: What is coffee and black coffee?
A: **Coffee**

Coffee is a widely‑consumed beverage made from the roasted seeds (often called “beans”) of the coffee plant (*Coffea* spp.). The process typically involves:

1. **Harvesting** – Ripe coffee cherries are picked, and the seeds are extracted.
2. **Pro...
⏱  2585ms | routed via Portkey gateway
──────────────────────────────────────────────────────────────

✅ Check portkey.ai → Logs to see both requ

### What changed?

```
Before:  Your App  →  Groq directly
After:   Your App  →  Portkey  →  Groq (via @flight-policsy integration)
```

- Portkey adds ~20–40ms then forwards to Groq using stored credentials
- Full request + response logged automatically — no extra code
- **You changed 3 lines. Your business logic is identical.**

---
# EXPERIMENT 2 — Metadata & Observability

**Goal:** Tag every request with user, session, and feature info so you can filter and analyse in the dashboard.

**New concept:** `portkey.with_options(metadata={...})` — attaches tags to a single request.
The special key `_user` powers per-user analytics.

In [14]:
section("EXP 2 — Metadata & Observability")

# new unique id

session = str(uuid.uuid4())[:8]

scenarios = [
    ("Abhay", "enterprise-rag",   "What is Kubernetes RBAC?"),
    ("Jay",   "docs-chatbot",     "How does BGP path selection work?"),
    ("Meet", "support-bot",      "What is SRIOV virtualization?"),
    ("alice", "enterprise-rag",   "Explain Kubernetes NetworkPolicy"),   # same user, diff Q
]




EXP 2 — Metadata & Observability


In [16]:
for user, feature, q in scenarios:
    t0= time.time()
    r = portkey.with_options(
        metadata={
        "_user": user,
        "session_id": session,
        "feature": feature,
        "enviroment":"notebook"
        }
    ).chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role":"user","content":q}
        ],max_tokens=300
    )
    ms = (time.time() - t0) * 1000
    print(f"\n👤 {user:8s} | 🔧 {feature:18s} | {ms:.0f}ms")
    print(f"  Q: {q}")
    print(f"  A: {r.choices[0].message.content[:120]}...")
    
    time.sleep(10)


👤 Abhay    | 🔧 enterprise-rag     | 2026ms
  Q: What is Kubernetes RBAC?
  A: **Kubernetes RBAC (Role‑Based Access Control)** is the native authorization mechanism that lets you define *who* can do ...

👤 Jay      | 🔧 docs-chatbot       | 1336ms
  Q: How does BGP path selection work?
  A: **Border Gateway Protocol (BGP) Path Selection – The Decision Process**

BGP is the “glue” that holds the Internet toget...

👤 Meet     | 🔧 support-bot        | 1433ms
  Q: What is SRIOV virtualization?
  A: ### SR‑IOV (Single Root I/O Virtualization) – an Overview

| Aspect | Description |
|--------|-------------|
| **What it...

👤 alice    | 🔧 enterprise-rag     | 1286ms
  Q: Explain Kubernetes NetworkPolicy
  A: ## Kubernetes NetworkPolicy – A High‑Level Overview

A **NetworkPolicy** is a Kubernetes resource that lets you control ...


### Why metadata matters in production

With `_user` tagging you can answer:
- *Which users generate the most cost?*
- *Which feature uses the most tokens?*
- *What did Alice's full session look like?*
- *Is the RAG pipeline slower than the support bot?*

All answered from the Portkey dashboard — no extra logging code.

---
# EXPERIMENT 3 — Automatic Retries

**Goal:** Portkey automatically retries failed requests with exponential backoff. App code never sees transient 429/5xx errors.

**New concept:** Pass a `config` dict to `Portkey(...)` with retry settings.

In [17]:
## how to config to portkey using code (API optional)

retry_config= {
    "retry":{
        "attempts":3,
        "on_status_code":[429,500,502]
    }
}

portkey_config=Portkey(api_key=PORTKEY_API_KEY,config=retry_config)

In [19]:
section("EXP 3 — Automatic Retries")
print("Config: 3 retry attempts on [429, 500, 502, 503, 504]")
print("Retries fire automatically on failure — transparent to your code\n")


try:
    t0 = time.time()
    r = portkey_config.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": "What is a AI?"}]
    )
    ms = (time.time() - t0) * 1000
    print(f"✅ Succeeded in {ms:.0f}ms")
    print(f"   {r.choices[0].message.content[:300]}")
    print("\nRetry sequence if Groq had failed:")
    print("  Attempt 1 → 429 → wait 1s → Attempt 2 → 429 → wait 2s → Attempt 3")
    print("  Your code only sees the final success or the last failure")
except Exception as e:
    print(f"❌ All attempts failed: {e}")


EXP 3 — Automatic Retries
Config: 3 retry attempts on [429, 500, 502, 503, 504]
Retries fire automatically on failure — transparent to your code

❌ All attempts failed: Error code: 400 - {'status': 'failure', 'error': {'code': 'inline_config_blocked', 'message': "Inline config is not allowed when block_inline_config is enabled. Reference a saved config by its 'pc-...' slug instead.", 'field': 'x-portkey-config'}, 'message': "Inline config is not allowed when block_inline_config is enabled. Reference a saved config by its 'pc-...' slug instead."}


---
# EXPERIMENT 4 — Request Timeouts

**Goal:** Kill requests that take too long. An LLM stall blocks your FastAPI worker indefinitely without a timeout.

**New concept:** `request_timeout` in milliseconds in the config dict. Portkey returns **HTTP 408** on timeout. Pair with fallbacks for auto-recovery.

In [20]:
timeout_config = {"request_timeout": 10000}   # 10 seconds in ms

portkey_timeout = Portkey(api_key=PORTKEY_API_KEY, config=timeout_config)

In [21]:
section("EXP 4 — Request Timeouts")
print("Timeout: 10,000ms (10 seconds). Portkey returns HTTP 408 if exceeded.\n")

try:
    t0 = time.time()
    r = portkey_timeout.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": "Explain Kubernetes networking in 2 sentences."}]
    )
    ms = (time.time() - t0) * 1000
    print(f"✅ Response in {ms:.0f}ms (within 10s timeout)")
    print(f"   {r.choices[0].message.content}")
except Exception as e:
    print(f"⏱  Timed out: {e}")
    print("   Portkey issued a 408. Pair with fallback to auto-switch providers on timeout.")

print("\n--- Combining timeout + retry (production pattern) ---")
combined = {
    "request_timeout": 10000,
    "retry": {"attempts": 2, "on_status_codes": [408, 429, 503]}
}
print(json.dumps(combined, indent=2))


EXP 4 — Request Timeouts
Timeout: 10,000ms (10 seconds). Portkey returns HTTP 408 if exceeded.

⏱  Timed out: Error code: 400 - {'status': 'failure', 'error': {'code': 'inline_config_blocked', 'message': "Inline config is not allowed when block_inline_config is enabled. Reference a saved config by its 'pc-...' slug instead.", 'field': 'x-portkey-config'}, 'message': "Inline config is not allowed when block_inline_config is enabled. Reference a saved config by its 'pc-...' slug instead."}
   Portkey issued a 408. Pair with fallback to auto-switch providers on timeout.

--- Combining timeout + retry (production pattern) ---
{
  "request_timeout": 10000,
  "retry": {
    "attempts": 2,
    "on_status_codes": [
      408,
      429,
      503
    ]
  }
}


---
# EXPERIMENT 5 — Fallbacks

**Goal:** If the primary model fails, automatically switch to the fallback. Users never see the error.

**New concept:** `strategy.mode = "fallback"` with an ordered `targets` list. First target is primary, rest are fallbacks.

**This experiment has two parts:**
- **Part 1** — primary works fine (shows fallback is ready but not needed)
- **Part 2** — primary uses a deliberately invalid key → Groq returns 401 → **real fallback fires live**

In [23]:
fallback_config = {
    "strategy": {"mode": "fallback"},
    "targets": [
        {"override_params": {"model": GROQ_MODEL}},        # primary
        {"override_params": {"model": GROQ_MODEL_2}}   # fallback if primary fails
    ]
}


portkey_fallback = Portkey(api_key=PORTKEY_API_KEY, config=fallback_config)

In [24]:
section("EXP 5 — Fallback Routing")
print(f"Primary  : {GROQ_MODEL}")
print(f"Fallback : {GROQ_MODEL_2}\n")

fallback_questions = [
    "What is Intel Technology?",
    "Explain Kubernetes persistent volume claims.",
]

for q in fallback_questions:
    try:
        t0 = time.time()
        r = portkey_fallback.chat.completions.create(
            messages=[{"role": "user", "content": q}]
        )
        ms = (time.time() - t0) * 1000
        show(q, r.choices[0].message.content, ms, label="primary served")
    except Exception as e:
        print(f"❌ {e}")
        print("   → Check that both GROQ_SLUG and GROQ_SLUG_2 are set correctly in c04")




EXP 5 — Fallback Routing
Primary  : @gkey/openai/gpt-oss-120b
Fallback : @gkey/llama-3.1-8b-instant

❌ Error code: 400 - {'status': 'failure', 'error': {'code': 'inline_config_blocked', 'message': "Inline config is not allowed when block_inline_config is enabled. Reference a saved config by its 'pc-...' slug instead.", 'field': 'x-portkey-config'}, 'message': "Inline config is not allowed when block_inline_config is enabled. Reference a saved config by its 'pc-...' slug instead."}
   → Check that both GROQ_SLUG and GROQ_SLUG_2 are set correctly in c04
❌ Error code: 400 - {'status': 'failure', 'error': {'code': 'inline_config_blocked', 'message': "Inline config is not allowed when block_inline_config is enabled. Reference a saved config by its 'pc-...' slug instead.", 'field': 'x-portkey-config'}, 'message': "Inline config is not allowed when block_inline_config is enabled. Reference a saved config by its 'pc-...' slug instead."}
   → Check that both GROQ_SLUG and GROQ_SLUG_2 are set c

In [31]:
# ── Part 2: FORCED fallback — bad primary key triggers real failover ──
print("\n\n--- FORCED FALLBACK DEMO ---")
print("Primary target uses a deliberately invalid Groq API key.")
print("Groq returns 401 → Portkey detects non-2xx → fallback fires automatically.\n")

forced_fallback_config = {
    "strategy": {"mode": "fallback"},
    "targets": [
        {
            "provider": "groq",
            "api_key": "gsk_FAKE_INVALID_KEY_THIS_WILL_FAIL",   # bad key → 401 from Groq
            "override_params": {"model": "llama-3.3-70b-versatile"}
        },
        {"override_params": {"model": GROQ_MODEL_2}}         # real key via virtual key
    ]
}

portkey_forced = Portkey(api_key=PORTKEY_API_KEY, config=forced_fallback_config)

try:
    t0 = time.time()
    r = portkey_forced.chat.completions.create(
        messages=[{"role": "user", "content": "What is ge ai?"}]
    )
    ms = (time.time() - t0) * 1000
    print(f"✅ Got a response in {ms:.0f}ms despite the bad primary key!")
    print(f"   {r.choices[0].message.content[:250]}")
    print("\n→ Check Portkey Logs: attempt 1 shows FAILED (401), attempt 2 shows SUCCEEDED")
    print("   The fallback fired automatically — app code never saw the error.")
except Exception as e:
    print(f"❌ Both targets failed: {e}")



--- FORCED FALLBACK DEMO ---
Primary target uses a deliberately invalid Groq API key.
Groq returns 401 → Portkey detects non-2xx → fallback fires automatically.

❌ Both targets failed: Error code: 400 - {'status': 'failure', 'error': {'code': 'inline_config_blocked', 'message': "Inline config is not allowed when block_inline_config is enabled. Reference a saved config by its 'pc-...' slug instead.", 'field': 'x-portkey-config'}, 'message': "Inline config is not allowed when block_inline_config is enabled. Reference a saved config by its 'pc-...' slug instead."}


### Narrow the fallback trigger

Default fires on any non-2xx. Narrow it to avoid accidental fallbacks on bad requests:

```python
# Only fall back on rate limits and server errors
"strategy": {"mode": "fallback", "on_status_codes": [429, 503]}
```

---
# EXPERIMENT 6 — Load Balancing

**Goal:** Split traffic between providers by weight. Use for gradual migration, A/B testing, or cost control.

**New concept:** `strategy.mode = "loadbalance"` with `weight` per target. Portkey normalises weights to percentages.